# 07 — Evaluate all registered models

This is the central comparison notebook. It discovers completed runs, validates compatible mappings, selects best checkpoints, exports common COCO JSON, computes accuracy/error/calibration metrics, and profiles models on shared hardware.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

In [ ]:
from src.training.checkpointing import RunRegistry
import pandas as pd
registry = RunRegistry(paths)
runs = registry.list_available_runs(dataset_track="2class")
display(pd.DataFrame(runs)) if runs else print("No registered 2class checkpoints yet.")


In [ ]:
DATASET_TRACK = "2class"
MODELS = []
MAX_IMAGES = 2 if SMOKE_TEST else None
if SMOKE_TEST:
    print("SMOKE_TEST: registry discovery verified; evaluation requires a completed checkpoint.")
else:
    command = [sys.executable, "scripts/evaluate.py", "--drive-root", DRIVE_ROOT,
               "--dataset-track", DATASET_TRACK, "--best-per-model"]
    if MODELS:
        command.extend(["--models", *MODELS])
    if MAX_IMAGES is not None:
        command.extend(["--max-images", str(MAX_IMAGES)])
    subprocess.run(command, check=True)
    subprocess.run(
        [sys.executable, "scripts/create_results_manifest.py", "--drive-root", DRIVE_ROOT,
         "--dataset-track", DATASET_TRACK],
        check=True,
    )


## Efficiency profiles

Run the profiler for each selected run on the same runtime. It performs 100 warm-ups and 500 synchronized timed iterations by default. Batch-4/8 and ONNX/TensorRT are separate experiments and failures remain visible.

In [ ]:
selected = registry.list_available_runs(dataset_track=DATASET_TRACK)
for run in selected:
    print("Profile with:", f"python scripts/profile_model.py --drive-root '{DRIVE_ROOT}' --run-id {run['run_id']}")

## Resolution scaling and ablations

Use separate registered runs for 640/1024/1280 and controlled ablations. Do not resize predictions post hoc and call it a training-resolution comparison.